In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Warnings
import warnings
warnings.filterwarnings('ignore')

In [19]:
# Loading user_item matrix
rat_mat = np.load("../transformed_Data/user_item_matrix.npy")
# Loading user indices correspondance
user_corres = np.load("../transformed_Data/user_corres_matrix_index.npy")
# Loading movie indices correspondance
movie_corres = np.load("../transformed_Data/movie_corres_matrix_index.npy")
# loading correspondance between movie indices and movie titles
movies_titles = pd.read_csv("../transformed_Data/MovieID_title.csv")
# loading ratings
ratings = np.load("../transformed_Data/ratings_np.npy")

In [5]:
# sparsity of the ratings matrix:
sparsity = 1.0 - ( np.count_nonzero(rat_mat) / float(rat_mat.size) )
print(sparsity)

0.9431750381059136


### 1. Computation of cos similarity matrix and predictions

In [12]:
#cos similarity between user and items
from sklearn.metrics.pairwise import pairwise_distances
user_similarity = pairwise_distances(rat_mat, metric='cosine')
item_similarity = pairwise_distances(rat_mat.T, metric='cosine')

In [ ]:
def predict(ratings, similarity, type='user'):

    n_users = len(np.unique(ratings[:,0]))
    n_items = len(np.unique(ratings[:,1]))

    if type == 'user':
        
        mean_user_rating = ratings.mean(axis=1)#mean pour chauqe utilisateur (type = float)
        #np.newaxis pour convertir mean_user_rating de array de float en array d'array pour l'utiliser avec ratings
        #puis on a normalisé la var ratings (rating - E)
        ratings_diff = (ratings - mean_user_rating[:, np.newaxis]) #(type === array comme la var rating)
        pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
        
    elif type == 'item':
        pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)]) 
        
    x = np.zeros((n_users, n_items))
    for i in range(0,n_items):
        a=max(pred[:,i])
        b=min(pred[:,i])
        c=0
        d=5
        for j in range(0,n_users):
            x[j,i]=(pred[:,i][j]-(a-c))*d/(b-a+c)
    
    return x

In [25]:
user_prediction = predict(rat_mat, user_similarity, type='user')

UnboundLocalError: cannot access local variable 'pred' where it is not associated with a value